In [ ]:
# ============================================================
# CELL 1 - IMPORT LIBRARIES AND CONFIGURE PATHS
# Purpose:
# Initialize the payment duration rule reconstruction
# environment without modifying source transactions.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from scipy.stats import kruskal, mannwhitneyu

# ------------------------------------------------------------
# 1. Locate the project root
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (PROJECT_ROOT / "data").is_dir(), (
    "Project root directory was not found."
)

# ------------------------------------------------------------
# 2. Define the cleaned dataset
# ------------------------------------------------------------

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "cleaned"
    / "swift_payments_cleaned.csv"
)

assert DATA_PATH.is_file(), (
    f"Cleaned dataset not found: {DATA_PATH}"
)

# ------------------------------------------------------------
# 3. Define output directories
# ------------------------------------------------------------

TABLE_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
    / "duration"
)

CHART_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "charts"
    / "duration"
)

RULE_DIR = PROJECT_ROOT / "outputs" / "rules"

for directory in [TABLE_DIR, CHART_DIR, RULE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 4. Configure analysis
# ------------------------------------------------------------

CHUNK_SIZE = 25_000
EXPECTED_RECORDS = 1_048_575
MIN_GROUP_SIZE = 100
RANDOM_STATE = 42

plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["font.size"] = 11

# ------------------------------------------------------------
# 5. Display setup
# ------------------------------------------------------------

print("=" * 65)
print("SWIFT DATATHON - PAYMENT DURATION INVESTIGATION")
print("=" * 65)

print("Dataset found:", DATA_PATH.exists())
print("Chunk size:", CHUNK_SIZE)
print("Expected records:", f"{EXPECTED_RECORDS:,}")
print("Table directory:", TABLE_DIR)
print("Chart directory:", CHART_DIR)

print("\nDuration analysis environment ready.")

In [ ]:
# ============================================================
# CELL 2 - SCHEMA VALIDATION AND HELPER FUNCTIONS
# Purpose:
# Validate required fields and define reusable
# duration analysis functions.
# ============================================================

REQUIRED_COLUMNS = [
    "EPOCH_SECONDS_NUM",
    "TRANSACTION_TYPE_CODE_STD",
    "PARTICIPANT_COUNT_NUM",
    "FX_COUNT_NUM",
    "TOTAL_DURATION_NUM",
    "USD_AMOUNT_NUM",
    "CURRENCY_CODE_STD",
    "PARTY_PAIR",
    "HAS_PATH_GAP"
]

available_columns = pd.read_csv(
    DATA_PATH,
    nrows=0
).columns.tolist()

missing_columns = [
    col for col in REQUIRED_COLUMNS
    if col not in available_columns
]

assert not missing_columns, (
    f"Missing analytical fields: {missing_columns}"
)

# ------------------------------------------------------------
# Define reusable helpers
# ------------------------------------------------------------

def export_table(df, filename):

    path = TABLE_DIR / filename

    df.to_csv(
        path,
        index=False,
        encoding="utf-8-sig"
    )

    print("Table exported:", filename)


def export_chart(filename):

    path = CHART_DIR / filename

    plt.savefig(
        path,
        dpi=200,
        bbox_inches="tight"
    )

    print("Chart exported:", filename)


def safe_ratio(numerator, denominator):

    if denominator == 0:
        return np.nan

    return numerator / denominator


def summarize_duration(df, group_columns):

    result = (
        df.groupby(
            group_columns,
            observed=True,
            dropna=False
        )
        .agg(
            TRANSACTION_COUNT=("DURATION", "size"),
            MEAN_DURATION=("DURATION", "mean"),
            MEDIAN_DURATION=("DURATION", "median"),
            P90_DURATION=(
                "DURATION",
                lambda x: x.quantile(0.90)
            ),
            P95_DURATION=(
                "DURATION",
                lambda x: x.quantile(0.95)
            ),
            P99_DURATION=(
                "DURATION",
                lambda x: x.quantile(0.99)
            )
        )
        .reset_index()
    )

    return result


print("Required columns:", len(REQUIRED_COLUMNS))
print("Missing columns:", missing_columns)
print("Schema validation passed.")
print("Duration helper functions initialized.")

In [ ]:
# ============================================================
# CELL 3 - LOAD COMPACT ANALYTICAL DATASET
# Purpose:
# Load only essential duration-related variables with
# chunk processing and validate data types.
# ============================================================

analysis_parts = []

loaded_rows = 0

reader = pd.read_csv(
    DATA_PATH,
    usecols=REQUIRED_COLUMNS,
    dtype=str,
    keep_default_na=False,
    na_filter=False,
    chunksize=CHUNK_SIZE
)

for chunk_number, chunk in enumerate(reader, start=1):

    # --------------------------------------------------------
    # 1. Convert numeric fields
    # --------------------------------------------------------

    numeric_columns = {
        "EPOCH_SECONDS_NUM": "EPOCH",
        "PARTICIPANT_COUNT_NUM": "PARTICIPANTS",
        "FX_COUNT_NUM": "FX_COUNT",
        "TOTAL_DURATION_NUM": "DURATION",
        "USD_AMOUNT_NUM": "AMOUNT"
    }

    for source, target in numeric_columns.items():

        chunk[target] = pd.to_numeric(
            chunk[source],
            errors="coerce"
        )

    # --------------------------------------------------------
    # 2. Standardize categorical analysis fields
    # --------------------------------------------------------

    chunk["ROUTING_TYPE"] = (
        chunk["TRANSACTION_TYPE_CODE_STD"]
        .str.strip()
        .str.upper()
    )

    chunk["CURRENCY"] = (
        chunk["CURRENCY_CODE_STD"]
        .str.strip()
        .str.upper()
    )

    chunk["PARTY_PAIR"] = (
        chunk["PARTY_PAIR"]
        .str.strip()
        .str.upper()
    )

    chunk["PATH_GAP"] = (
        chunk["HAS_PATH_GAP"]
        .str.strip()
        .str.upper()
    )

    # --------------------------------------------------------
    # 3. Keep only required analytical fields
    # --------------------------------------------------------

    compact = chunk[
        [
            "EPOCH",
            "ROUTING_TYPE",
            "PARTICIPANTS",
            "FX_COUNT",
            "DURATION",
            "AMOUNT",
            "CURRENCY",
            "PARTY_PAIR",
            "PATH_GAP"
        ]
    ].copy()

    analysis_parts.append(compact)

    loaded_rows += len(compact)

    if chunk_number % 10 == 0:
        print(
            f"Chunk {chunk_number}: "
            f"{loaded_rows:,} records loaded"
        )

# ------------------------------------------------------------
# 4. Combine compact chunks
# ------------------------------------------------------------

duration_df = pd.concat(
    analysis_parts,
    ignore_index=True
)

del analysis_parts

# ------------------------------------------------------------
# 5. Optimize categorical data types
# ------------------------------------------------------------

for col in [
    "ROUTING_TYPE",
    "CURRENCY",
    "PARTY_PAIR",
    "PATH_GAP"
]:
    duration_df[col] = (
        duration_df[col].astype("category")
    )

# ------------------------------------------------------------
# 6. Validate record count and numeric integrity
# ------------------------------------------------------------

assert len(duration_df) == EXPECTED_RECORDS, (
    "Transaction count mismatch."
)

numeric_validation = pd.DataFrame([
    {
        "COLUMN": col,
        "MISSING_VALUES": int(duration_df[col].isna().sum()),
        "NEGATIVE_VALUES": int((duration_df[col] < 0).sum())
    }
    for col in [
        "EPOCH",
        "PARTICIPANTS",
        "FX_COUNT",
        "DURATION",
        "AMOUNT"
    ]
])

display(numeric_validation)

assert numeric_validation["MISSING_VALUES"].sum() == 0
assert numeric_validation["NEGATIVE_VALUES"].sum() == 0

print("\nCompact dataset loaded successfully.")
print("Transactions:", f"{len(duration_df):,}")

print(
    "Memory usage (MB):",
    round(
        duration_df.memory_usage(deep=True).sum()
        / (1024 ** 2),
        2
    )
)

In [ ]:
# ============================================================
# CELL 4 - GLOBAL PAYMENT DURATION ANALYSIS
# Purpose:
# Analyze overall payment duration, calculate descriptive
# statistics, investigate slow-payment thresholds, and
# visualize the duration distribution.
#
# Unit: Seconds
# Data Protection:
# - No transaction records are removed.
# - Original duration values remain unchanged.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick


# ------------------------------------------------------------
# 1. Prepare duration values
# ------------------------------------------------------------

duration_values = (
    duration_df["DURATION"]
    .to_numpy(dtype=np.float64)
)

assert len(duration_values) == EXPECTED_RECORDS, (
    "Duration record count does not match the expected value."
)

assert np.isfinite(duration_values).all(), (
    "Invalid duration values detected."
)

assert (duration_values >= 0).all(), (
    "Negative duration values detected."
)


# ------------------------------------------------------------
# 2. Calculate global duration statistics
# ------------------------------------------------------------

global_stats = {
    "TRANSACTIONS": int(len(duration_values)),
    "MEAN_DURATION": float(np.mean(duration_values)),
    "MEDIAN_DURATION": float(np.median(duration_values)),
    "STD_DURATION": float(
        np.std(duration_values, ddof=1)
    ),
    "MIN_DURATION": float(np.min(duration_values)),
    "P25_DURATION": float(
        np.percentile(duration_values, 25)
    ),
    "P75_DURATION": float(
        np.percentile(duration_values, 75)
    ),
    "P90_DURATION": float(
        np.percentile(duration_values, 90)
    ),
    "P95_DURATION": float(
        np.percentile(duration_values, 95)
    ),
    "P99_DURATION": float(
        np.percentile(duration_values, 99)
    ),
    "MAX_DURATION": float(np.max(duration_values))
}

global_duration_table = pd.DataFrame([
    {
        "METRIC": metric,
        "VALUE": value,
        "UNIT": (
            "Transactions"
            if metric == "TRANSACTIONS"
            else "Seconds"
        )
    }
    for metric, value in global_stats.items()
])

export_table(
    global_duration_table,
    "duration_global_statistics.csv"
)


# ------------------------------------------------------------
# 3. Calculate slow-payment threshold statistics
# ------------------------------------------------------------

slow_thresholds = {
    "ABOVE_60_SECONDS": 60,
    "ABOVE_300_SECONDS": 300,
    "ABOVE_900_SECONDS": 900,
    "ABOVE_1800_SECONDS": 1800
}

tail_rows = []

for label, threshold in slow_thresholds.items():

    slow_count = int(
        (duration_values > threshold).sum()
    )

    slow_percentage = (
        slow_count / len(duration_values) * 100
    )

    tail_rows.append({
        "THRESHOLD": label,
        "THRESHOLD_SECONDS": threshold,
        "TRANSACTION_COUNT": slow_count,
        "PERCENT_OF_ALL": slow_percentage
    })

tail_summary = pd.DataFrame(tail_rows)

export_table(
    tail_summary,
    "duration_tail_thresholds.csv"
)


# ------------------------------------------------------------
# 4. Display global statistical results
# ------------------------------------------------------------

print("=" * 65)
print("GLOBAL PAYMENT DURATION STATISTICS")
print("=" * 65)

display(global_duration_table)

print("\nSlow Payment Threshold Statistics:")
display(tail_summary)


# ------------------------------------------------------------
# 5. Prepare positive duration values for log visualization
# ------------------------------------------------------------

positive_durations = duration_values[
    duration_values > 0
]

assert len(positive_durations) > 0, (
    "No positive duration values available."
)


# ------------------------------------------------------------
# 6. Create logarithmic histogram bins
# ------------------------------------------------------------

bins = np.geomspace(
    positive_durations.min(),
    positive_durations.max(),
    80
)


# ------------------------------------------------------------
# 7. Create payment duration histogram
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(12, 6))

ax.hist(
    positive_durations,
    bins=bins,
    edgecolor="white",
    alpha=0.85
)


# ------------------------------------------------------------
# 8. Configure readable axis labels in seconds
# ------------------------------------------------------------

ax.set_xscale("log")

tick_values = [
    2, 5, 10, 20, 50, 100,
    200, 500, 1000, 2000,
    5000, 10000
]

visible_ticks = [
    value
    for value in tick_values
    if positive_durations.min()
    <= value
    <= positive_durations.max()
]

ax.set_xticks(visible_ticks)

ax.xaxis.set_major_formatter(
    mtick.FuncFormatter(
        lambda value, position: f"{value:,.0f}"
    )
)

ax.xaxis.set_minor_formatter(
    mtick.NullFormatter()
)

ax.tick_params(
    axis="x",
    labelrotation=45
)

ax.yaxis.set_major_formatter(
    mtick.StrMethodFormatter("{x:,.0f}")
)


# ------------------------------------------------------------
# 9. Add median duration reference line
# ------------------------------------------------------------

median_duration = global_stats["MEDIAN_DURATION"]

ax.axvline(
    median_duration,
    linestyle="--",
    linewidth=2,
    label=f"Median: {median_duration:,.1f} seconds"
)


# ------------------------------------------------------------
# 10. Configure chart presentation
# ------------------------------------------------------------

ax.set_title(
    "Overall Payment Duration Distribution",
    fontsize=15,
    fontweight="bold"
)

ax.set_xlabel(
    "Payment Duration (Seconds, Logarithmic Axis)"
)

ax.set_ylabel(
    "Transaction Count"
)

ax.legend()

ax.grid(
    axis="y",
    alpha=0.2
)

plt.tight_layout()


# ------------------------------------------------------------
# 11. Export the chart
# ------------------------------------------------------------

export_chart(
    "duration_global_distribution.png"
)

plt.show()


# ------------------------------------------------------------
# 12. Final validation
# ------------------------------------------------------------

assert global_stats["TRANSACTIONS"] == EXPECTED_RECORDS

assert (
    global_stats["MIN_DURATION"]
    <= global_stats["MEDIAN_DURATION"]
    <= global_stats["MAX_DURATION"]
)

print("\n" + "=" * 65)
print("GLOBAL DURATION ANALYSIS COMPLETED")
print("=" * 65)

print(
    "Total transactions:",
    f"{global_stats['TRANSACTIONS']:,}"
)

print(
    "Median duration:",
    f"{global_stats['MEDIAN_DURATION']:,.1f} seconds"
)

print(
    "Mean duration:",
    f"{global_stats['MEAN_DURATION']:,.2f} seconds"
)

print("\nDuration unit: Seconds")
print("Axis spacing: Logarithmic")
print("Original transaction values: Unchanged")

print("\nGlobal duration analysis completed successfully.")

In [ ]:
# ============================================================
# OPTIONAL - LINEAR DURATION HISTOGRAM (0 TO 300 SECONDS)
# Purpose:
# Show short payment durations using a conventional axis.
# ============================================================

short_durations = duration_df.loc[
    duration_df["DURATION"].between(0, 300),
    "DURATION"
]

fig, ax = plt.subplots(figsize=(11, 6))

ax.hist(
    short_durations,
    bins=np.arange(0, 305, 5),
    edgecolor="white"
)

ax.axvline(
    global_stats["MEDIAN_DURATION"],
    linestyle="--",
    linewidth=2,
    label=(
        f"Overall Median: "
        f"{global_stats['MEDIAN_DURATION']:.1f} seconds"
    )
)

ax.set_title(
    "Payment Duration Distribution (0–300 Seconds)",
    fontsize=15,
    fontweight="bold"
)

ax.set_xlabel("Payment Duration (Seconds)")
ax.set_ylabel("Transaction Count")

ax.set_xlim(0, 300)

ax.yaxis.set_major_formatter(
    mtick.StrMethodFormatter("{x:,.0f}")
)

ax.legend()

plt.tight_layout()

export_chart(
    "duration_short_range.png"
)

plt.show()

In [ ]:
# ============================================================
# CELL 5 - PAYMENT DURATION BY ROUTING TYPE
# Purpose:
# Compare duration distributions across Direct, Indirect,
# and Covered routes.
# ============================================================

route_duration = summarize_duration(
    duration_df,
    ["ROUTING_TYPE"]
)

route_duration = route_duration.sort_values(
    "MEDIAN_DURATION"
)

# ------------------------------------------------------------
# 1. Compare with Direct baseline
# ------------------------------------------------------------

direct_row = route_duration.loc[
    route_duration["ROUTING_TYPE"] == "DIRECT"
]

assert len(direct_row) == 1

direct_median = float(
    direct_row["MEDIAN_DURATION"].iloc[0]
)

route_duration["MEDIAN_VS_DIRECT_RATIO"] = (
    route_duration["MEDIAN_DURATION"] / direct_median
)

route_duration["MEDIAN_VS_DIRECT_DIFFERENCE"] = (
    route_duration["MEDIAN_DURATION"] - direct_median
)

export_table(
    route_duration,
    "duration_by_routing_type.csv"
)

display(route_duration)

# ------------------------------------------------------------
# 2. Create median comparison chart
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(
    route_duration["ROUTING_TYPE"].astype(str),
    route_duration["MEDIAN_DURATION"]
)

ax.set_title(
    "Median Payment Duration by Routing Type",
    fontweight="bold"
)

ax.set_xlabel("Routing Type")
ax.set_ylabel("Median Duration (Seconds)")

ax.set_ylim(
    0,
    route_duration["MEDIAN_DURATION"].max() * 1.2
)

for bar in bars:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{bar.get_height():.1f}s",
        ha="center",
        va="bottom"
    )

plt.tight_layout()

export_chart("duration_median_by_routing.png")

plt.show()

print("Routing duration comparison completed.")

In [ ]:
# ============================================================
# CELL 6 - PARTICIPANT COUNT AND ROUTING TYPE
# Purpose:
# Compare duration across participant counts within
# each routing category.
# ============================================================

participant_route_duration = summarize_duration(
    duration_df,
    ["ROUTING_TYPE", "PARTICIPANTS"]
)

participant_route_duration["SAMPLE_STATUS"] = np.where(
    participant_route_duration["TRANSACTION_COUNT"]
    >= MIN_GROUP_SIZE,
    "ADEQUATE_FOR_EXPLORATION",
    "SMALL_SAMPLE"
)

participant_route_duration = (
    participant_route_duration.sort_values(
        ["ROUTING_TYPE", "PARTICIPANTS"]
    )
)

export_table(
    participant_route_duration,
    "duration_by_routing_and_participants.csv"
)

display(participant_route_duration)

# ------------------------------------------------------------
# 1. Focus on groups with sufficient sample sizes
# ------------------------------------------------------------

plot_data = participant_route_duration.loc[
    (participant_route_duration["TRANSACTION_COUNT"]
     >= MIN_GROUP_SIZE)
    &
    (participant_route_duration["PARTICIPANTS"] <= 10)
].copy()

fig, ax = plt.subplots(figsize=(11, 6))

for route, group in plot_data.groupby(
    "ROUTING_TYPE",
    observed=True
):

    group = group.sort_values("PARTICIPANTS")

    ax.plot(
        group["PARTICIPANTS"],
        group["MEDIAN_DURATION"],
        marker="o",
        linewidth=2,
        label=str(route)
    )

ax.set_title(
    "Median Duration by Routing Type and Participant Count",
    fontweight="bold"
)

ax.set_xlabel("Participant Count")
ax.set_ylabel("Median Duration (Seconds)")

ax.grid(alpha=0.2)
ax.legend(title="Routing Type")

plt.tight_layout()

export_chart("duration_routing_participant.png")

plt.show()

print("Participant-adjusted routing exploration completed.")

In [ ]:
# ============================================================
# CELL 7 - CONDITIONAL FX DURATION ANALYSIS
# Purpose:
# Investigate FX-duration relationships within matched
# routing type and participant count groups.
# ============================================================

fx_duration = summarize_duration(
    duration_df,
    ["ROUTING_TYPE", "PARTICIPANTS", "FX_COUNT"]
)

fx_duration["SAMPLE_STATUS"] = np.where(
    fx_duration["TRANSACTION_COUNT"] >= MIN_GROUP_SIZE,
    "ADEQUATE_FOR_EXPLORATION",
    "SMALL_SAMPLE"
)

fx_duration = fx_duration.sort_values(
    ["ROUTING_TYPE", "PARTICIPANTS", "FX_COUNT"]
)

export_table(
    fx_duration,
    "duration_by_routing_participants_fx.csv"
)

# ------------------------------------------------------------
# 1. Compare adjacent FX counts within matching groups
# ------------------------------------------------------------

comparison_rows = []

for (route, participants), group in fx_duration.groupby(
    ["ROUTING_TYPE", "PARTICIPANTS"],
    observed=True
):

    group = group.sort_values("FX_COUNT")

    records = {
        int(row.FX_COUNT): row
        for row in group.itertuples(index=False)
    }

    for fx_value, current in records.items():

        next_fx = fx_value + 1

        if next_fx not in records:
            continue

        following = records[next_fx]

        comparison_rows.append({
            "ROUTING_TYPE": str(route),
            "PARTICIPANTS": participants,
            "BASE_FX": fx_value,
            "NEXT_FX": next_fx,
            "BASE_COUNT": current.TRANSACTION_COUNT,
            "NEXT_COUNT": following.TRANSACTION_COUNT,
            "BASE_MEDIAN": current.MEDIAN_DURATION,
            "NEXT_MEDIAN": following.MEDIAN_DURATION,
            "MEDIAN_DIFFERENCE": (
                following.MEDIAN_DURATION
                - current.MEDIAN_DURATION
            ),
            "MEDIAN_RATIO": safe_ratio(
                following.MEDIAN_DURATION,
                current.MEDIAN_DURATION
            )
        })

fx_adjacent_comparison = pd.DataFrame(comparison_rows)

fx_adjacent_comparison["RELIABLE_COMPARISON"] = (
    (fx_adjacent_comparison["BASE_COUNT"] >= MIN_GROUP_SIZE)
    &
    (fx_adjacent_comparison["NEXT_COUNT"] >= MIN_GROUP_SIZE)
)

export_table(
    fx_adjacent_comparison,
    "fx_adjacent_duration_comparisons.csv"
)

print("Conditional FX duration statistics:")
display(fx_duration.head(40))

print("\nAdjacent FX comparisons with sufficient samples:")
display(
    fx_adjacent_comparison.loc[
        fx_adjacent_comparison["RELIABLE_COMPARISON"]
    ].head(40)
)

print("Conditional FX investigation completed.")

In [ ]:
# ============================================================
# CELL 8 - MATCHED INDIRECT VS COVERED DURATION
# Purpose:
# Compare non-direct routing categories within matching
# participant, FX, currency, and party segments.
# ============================================================

matched_data = duration_df.loc[
    duration_df["ROUTING_TYPE"].isin(
        ["INDIRECT", "COVERED"]
    ),
    [
        "ROUTING_TYPE",
        "PARTICIPANTS",
        "FX_COUNT",
        "CURRENCY",
        "PARTY_PAIR",
        "DURATION"
    ]
].copy()

matched_data["LOG_DURATION"] = np.log1p(
    matched_data["DURATION"]
)

strata_columns = [
    "PARTICIPANTS",
    "FX_COUNT",
    "CURRENCY",
    "PARTY_PAIR"
]

# ------------------------------------------------------------
# 1. Aggregate duration within each comparable stratum
# ------------------------------------------------------------

strata_summary = (
    matched_data.groupby(
        strata_columns + ["ROUTING_TYPE"],
        observed=True
    )
    .agg(
        COUNT=("DURATION", "size"),
        MEAN_LOG_DURATION=("LOG_DURATION", "mean"),
        MEDIAN_DURATION=("DURATION", "median")
    )
    .reset_index()
)

# ------------------------------------------------------------
# 2. Prepare matched routing values
# ------------------------------------------------------------

matched_pivot = strata_summary.pivot(
    index=strata_columns,
    columns="ROUTING_TYPE",
    values=[
        "COUNT",
        "MEAN_LOG_DURATION",
        "MEDIAN_DURATION"
    ]
)

matched_pivot.columns = [
    f"{metric}_{route}"
    for metric, route in matched_pivot.columns
]

matched_pivot = matched_pivot.reset_index()

required_matched_columns = [
    "COUNT_INDIRECT",
    "COUNT_COVERED",
    "MEAN_LOG_DURATION_INDIRECT",
    "MEAN_LOG_DURATION_COVERED"
]

for col in required_matched_columns:
    assert col in matched_pivot.columns

# ------------------------------------------------------------
# 3. Retain strata represented by both routing categories
# ------------------------------------------------------------

matched_pivot = matched_pivot.dropna(
    subset=required_matched_columns
).copy()

matched_pivot["LOG_DURATION_DIFFERENCE"] = (
    matched_pivot["MEAN_LOG_DURATION_COVERED"]
    - matched_pivot["MEAN_LOG_DURATION_INDIRECT"]
)

matched_pivot["MATCH_WEIGHT"] = np.minimum(
    matched_pivot["COUNT_INDIRECT"],
    matched_pivot["COUNT_COVERED"]
)

matched_pivot["RELATIVE_DIFFERENCE_PERCENT"] = (
    np.expm1(
        matched_pivot["LOG_DURATION_DIFFERENCE"]
    ) * 100
)

# ------------------------------------------------------------
# 4. Calculate an overall matched comparison
# ------------------------------------------------------------

total_match_weight = matched_pivot["MATCH_WEIGHT"].sum()

assert total_match_weight > 0, (
    "No comparable Indirect and Covered strata were found."
)

weighted_log_difference = np.average(
    matched_pivot["LOG_DURATION_DIFFERENCE"],
    weights=matched_pivot["MATCH_WEIGHT"]
)

overall_relative_difference = (
    np.expm1(weighted_log_difference) * 100
)

matched_summary = pd.DataFrame([{
    "MATCHED_STRATA": len(matched_pivot),
    "MATCH_WEIGHT_TOTAL": total_match_weight,
    "WEIGHTED_LOG_DIFFERENCE": weighted_log_difference,
    "RELATIVE_DIFFERENCE_PERCENT": overall_relative_difference
}])

# ------------------------------------------------------------
# 5. Export results
# ------------------------------------------------------------

export_table(
    matched_pivot,
    "indirect_covered_duration_matched_strata.csv"
)

export_table(
    matched_summary,
    "indirect_covered_duration_matched_summary.csv"
)

print("Matched comparison summary:")
display(matched_summary)

print("\nLargest well-supported matched strata:")
display(
    matched_pivot.sort_values(
        "MATCH_WEIGHT",
        ascending=False
    ).head(20)
)

print("Matched Indirect/Covered duration analysis completed.")

In [ ]:
# ============================================================
# CELL 9 - ROUTING GAP AND PAYMENT DURATION
# Purpose:
# Compare duration by path observability within routing,
# participant count, and FX count groups.
# ============================================================

gap_duration = summarize_duration(
    duration_df,
    [
        "ROUTING_TYPE",
        "PARTICIPANTS",
        "FX_COUNT",
        "PATH_GAP"
    ]
)

export_table(
    gap_duration,
    "duration_by_routing_participants_fx_gap.csv"
)

# ------------------------------------------------------------
# 1. Create comparable gap/non-gap strata
# ------------------------------------------------------------

gap_pivot = gap_duration.pivot(
    index=[
        "ROUTING_TYPE",
        "PARTICIPANTS",
        "FX_COUNT"
    ],
    columns="PATH_GAP",
    values=[
        "TRANSACTION_COUNT",
        "MEDIAN_DURATION"
    ]
)

gap_pivot.columns = [
    f"{metric}_{gap}"
    for metric, gap in gap_pivot.columns
]

gap_pivot = gap_pivot.reset_index()

required_gap_columns = [
    "TRANSACTION_COUNT_Y",
    "TRANSACTION_COUNT_N",
    "MEDIAN_DURATION_Y",
    "MEDIAN_DURATION_N"
]

for col in required_gap_columns:
    assert col in gap_pivot.columns

gap_pivot = gap_pivot.dropna(
    subset=required_gap_columns
).copy()

# ------------------------------------------------------------
# 2. Compare medians within matched groups
# ------------------------------------------------------------

gap_pivot["MEDIAN_DIFFERENCE"] = (
    gap_pivot["MEDIAN_DURATION_Y"]
    - gap_pivot["MEDIAN_DURATION_N"]
)

gap_pivot["MEDIAN_RATIO"] = (
    gap_pivot["MEDIAN_DURATION_Y"]
    / gap_pivot["MEDIAN_DURATION_N"]
)

gap_pivot["SAMPLE_STATUS"] = np.where(
    (
        gap_pivot["TRANSACTION_COUNT_Y"] >= MIN_GROUP_SIZE
    )
    &
    (
        gap_pivot["TRANSACTION_COUNT_N"] >= MIN_GROUP_SIZE
    ),
    "ADEQUATE_FOR_EXPLORATION",
    "SMALL_SAMPLE"
)

export_table(
    gap_pivot,
    "gap_duration_matched_comparison.csv"
)

display(
    gap_pivot.loc[
        gap_pivot["SAMPLE_STATUS"]
        == "ADEQUATE_FOR_EXPLORATION"
    ].head(40)
)

print("Routing gap duration comparison completed.")

In [ ]:
# ============================================================
# CELL 10 - SLOW PAYMENT INVESTIGATION
# Purpose:
# Identify slow-payment concentration across routing
# types and participant counts.
# ============================================================

SLOW_THRESHOLDS = [300, 900, 1800]

slow_records = []

group_columns = [
    "ROUTING_TYPE",
    "PARTICIPANTS"
]

for threshold in SLOW_THRESHOLDS:

    temp = (
        duration_df.groupby(
            group_columns,
            observed=True
        )["DURATION"]
        .agg(
            TOTAL="size",
            SLOW_COUNT=lambda x: int(
                (x > threshold).sum()
            )
        )
        .reset_index()
    )

    temp["THRESHOLD_SECONDS"] = threshold

    temp["SLOW_RATE_PERCENT"] = (
        temp["SLOW_COUNT"] / temp["TOTAL"] * 100
    )

    slow_records.append(temp)

slow_payment_table = pd.concat(
    slow_records,
    ignore_index=True
)

slow_payment_table["SAMPLE_STATUS"] = np.where(
    slow_payment_table["TOTAL"] >= MIN_GROUP_SIZE,
    "ADEQUATE_FOR_EXPLORATION",
    "SMALL_SAMPLE"
)

export_table(
    slow_payment_table,
    "slow_payment_rates_by_route_participants.csv"
)

print("Slow payment rates:")
display(
    slow_payment_table.loc[
        slow_payment_table["SAMPLE_STATUS"]
        == "ADEQUATE_FOR_EXPLORATION"
    ].head(40)
)

# ------------------------------------------------------------
# 1. Compare slow-payment rates by routing type
# ------------------------------------------------------------

route_slow = []

for threshold in SLOW_THRESHOLDS:

    grouped = (
        duration_df.groupby(
            "ROUTING_TYPE",
            observed=True
        )["DURATION"]
        .agg(
            TOTAL="size",
            SLOW_COUNT=lambda x: int(
                (x > threshold).sum()
            )
        )
        .reset_index()
    )

    grouped["THRESHOLD_SECONDS"] = threshold

    grouped["SLOW_RATE_PERCENT"] = (
        grouped["SLOW_COUNT"]
        / grouped["TOTAL"] * 100
    )

    route_slow.append(grouped)

route_slow_table = pd.concat(
    route_slow,
    ignore_index=True
)

export_table(
    route_slow_table,
    "slow_payment_rates_by_routing.csv"
)

display(route_slow_table)

print("Slow payment analysis completed.")

In [ ]:
# ============================================================
# CELL 11 - DIRECT PAYMENT DURATION REGIMES
# Purpose:
# Investigate why some two-participant Direct payments
# still show long processing durations.
# ============================================================

direct_df = duration_df.loc[
    duration_df["ROUTING_TYPE"] == "DIRECT"
].copy()

assert len(direct_df) == 662_245

# ------------------------------------------------------------
# 1. Compare FX groups within Direct routing
# ------------------------------------------------------------

direct_fx_duration = summarize_duration(
    direct_df,
    ["FX_COUNT"]
)

export_table(
    direct_fx_duration,
    "direct_duration_by_fx.csv"
)

display(direct_fx_duration)

# ------------------------------------------------------------
# 2. Define exploratory duration bands
# ------------------------------------------------------------

duration_bins = [
    -np.inf,
    60,
    300,
    900,
    np.inf
]

duration_labels = [
    "0-60s",
    "60-300s",
    "300-900s",
    "900s+"
]

direct_df["DURATION_BAND"] = pd.cut(
    direct_df["DURATION"],
    bins=duration_bins,
    labels=duration_labels
)

direct_band_summary = (
    direct_df.groupby(
        ["DURATION_BAND", "FX_COUNT"],
        observed=True
    )
    .agg(
        TRANSACTION_COUNT=("DURATION", "size"),
        MEDIAN_AMOUNT=("AMOUNT", "median"),
        MEDIAN_DURATION=("DURATION", "median")
    )
    .reset_index()
)

export_table(
    direct_band_summary,
    "direct_duration_regime_summary.csv"
)

display(direct_band_summary)

# ------------------------------------------------------------
# 3. Compare direct-payment duration quantiles by currency
# ------------------------------------------------------------

currency_counts = (
    direct_df["CURRENCY"].value_counts()
)

major_currencies = currency_counts[
    currency_counts >= 1000
].index

direct_currency = summarize_duration(
    direct_df.loc[
        direct_df["CURRENCY"].isin(major_currencies)
    ],
    ["CURRENCY"]
)

direct_currency = direct_currency.sort_values(
    "TRANSACTION_COUNT",
    ascending=False
)

export_table(
    direct_currency,
    "direct_duration_by_currency.csv"
)

print("Major currency duration profiles:")
display(direct_currency.head(20))

# ------------------------------------------------------------
# 4. Compare log-duration distribution by FX count
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(11, 6))

for fx, group in direct_df.groupby(
    "FX_COUNT",
    observed=True
):

    values = group["DURATION"].to_numpy()

    if len(values) < 100:
        continue

    values = np.sort(values)

    cumulative = (
        np.arange(1, len(values) + 1) / len(values)
    )

    ax.plot(
        values,
        cumulative,
        label=f"FX={fx}"
    )

ax.set_xscale("log")

ax.set_title(
    "Direct Payment Duration ECDF by FX Count",
    fontweight="bold"
)

ax.set_xlabel("Duration (Seconds, Log Scale)")
ax.set_ylabel("Cumulative Proportion")

ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()

export_chart("direct_duration_ecdf_by_fx.png")

plt.show()

print("Direct payment regime investigation completed.")

In [ ]:
# ============================================================
# CELL 12 - CURRENCY, PARTY AND TIME CHECKS
# Purpose:
# Explore alternative explanations for duration patterns.
# ============================================================

# ------------------------------------------------------------
# 1. Currency within routing type
# ------------------------------------------------------------

currency_route_duration = summarize_duration(
    duration_df,
    ["ROUTING_TYPE", "CURRENCY"]
)

currency_route_duration["SAMPLE_STATUS"] = np.where(
    currency_route_duration["TRANSACTION_COUNT"] >= 500,
    "ADEQUATE_FOR_EXPLORATION",
    "SMALL_SAMPLE"
)

export_table(
    currency_route_duration,
    "duration_by_route_currency.csv"
)

# ------------------------------------------------------------
# 2. Party relationship within routing type
# ------------------------------------------------------------

party_route_duration = summarize_duration(
    duration_df,
    ["ROUTING_TYPE", "PARTY_PAIR"]
)

export_table(
    party_route_duration,
    "duration_by_route_party.csv"
)

# ------------------------------------------------------------
# 3. Derive UTC hour
# ------------------------------------------------------------

datetime_utc = pd.to_datetime(
    duration_df["EPOCH"],
    unit="s",
    utc=True,
    errors="coerce"
)

duration_df["HOUR_UTC"] = (
    datetime_utc.dt.hour.astype("Int64")
)

duration_df["DAY_OF_WEEK_UTC"] = (
    datetime_utc.dt.day_name()
)

# ------------------------------------------------------------
# 4. Analyze duration by UTC hour and routing type
# ------------------------------------------------------------

hour_duration = summarize_duration(
    duration_df,
    ["ROUTING_TYPE", "HOUR_UTC"]
)

export_table(
    hour_duration,
    "duration_by_route_hour.csv"
)

# ------------------------------------------------------------
# 5. Display selected results
# ------------------------------------------------------------

print("Major currency comparisons:")
display(
    currency_route_duration.loc[
        currency_route_duration["SAMPLE_STATUS"]
        == "ADEQUATE_FOR_EXPLORATION"
    ].sort_values(
        ["ROUTING_TYPE", "TRANSACTION_COUNT"],
        ascending=[True, False]
    ).head(30)
)

print("\nParty relationship comparisons:")
display(party_route_duration)

print("\nUTC hour comparisons:")
display(hour_duration.head(30))

print("Alternative explanation screening completed.")

In [ ]:
# ============================================================
# CELL 13 - EXPLAINABLE DURATION DECISION TREE
# Purpose:
# Discover potential duration-generating conditions using
# a shallow and interpretable regression tree.
#
# Data Protection:
# - Original transaction records are not modified.
# - Sampling is used only for model exploration.
# - Model results are not treated as causal evidence.
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.metrics import mean_absolute_error, r2_score

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Create a reproducible sample
# ------------------------------------------------------------

sample_size = min(
    200_000,
    len(duration_df)
)

tree_data = duration_df.sample(
    n=sample_size,
    random_state=RANDOM_STATE
).copy()

print("=" * 65)
print("EXPLAINABLE PAYMENT DURATION DECISION TREE")
print("=" * 65)

print("Sample size:", f"{sample_size:,}")


# ------------------------------------------------------------
# 2. Prepare the target variable
# ------------------------------------------------------------

# Apply log1p to reduce the influence of long-tail durations.
# The original duration values remain unchanged.

tree_data["LOG_DURATION"] = np.log1p(
    tree_data["DURATION"]
)


# ------------------------------------------------------------
# 3. Create a log-transformed amount feature
# ------------------------------------------------------------

# Log-transformed amount is used only as a model feature.

tree_data["LOG_AMOUNT"] = np.log1p(
    tree_data["AMOUNT"]
)


# ------------------------------------------------------------
# 4. Create currency groups safely
# ------------------------------------------------------------

# Convert categorical currency values to strings before
# assigning the new "OTHER" category.

currency_values = (
    tree_data["CURRENCY"]
    .astype(str)
)

top_currencies = (
    currency_values
    .value_counts()
    .head(10)
    .index
)

tree_data["CURRENCY_GROUP"] = currency_values.where(
    currency_values.isin(top_currencies),
    "OTHER"
)

print("\nTop 10 currencies:")
print(list(top_currencies))

print("\nCurrency group distribution:")
print(
    tree_data["CURRENCY_GROUP"]
    .value_counts()
)


# ------------------------------------------------------------
# 5. Define explanatory features
# ------------------------------------------------------------

feature_columns = [
    "PARTICIPANTS",
    "FX_COUNT",
    "ROUTING_TYPE",
    "PATH_GAP",
    "PARTY_PAIR",
    "CURRENCY_GROUP",
    "LOG_AMOUNT"
]

X = tree_data[feature_columns].copy()

y = tree_data["LOG_DURATION"].copy()


# ------------------------------------------------------------
# 6. Convert categorical features using one-hot encoding
# ------------------------------------------------------------

categorical_columns = [
    "ROUTING_TYPE",
    "PATH_GAP",
    "PARTY_PAIR",
    "CURRENCY_GROUP"
]

# Convert categorical columns to strings for consistency.

for col in categorical_columns:
    X[col] = X[col].astype(str)

X = pd.get_dummies(
    X,
    columns=categorical_columns,
    dtype=int
)


# ------------------------------------------------------------
# 7. Validate model input
# ------------------------------------------------------------

assert not X.isna().any().any(), (
    "Missing values were detected in the model features."
)

assert not y.isna().any(), (
    "Missing values were detected in the target variable."
)

assert np.isfinite(X.to_numpy()).all(), (
    "Non-finite values were detected in model features."
)

assert np.isfinite(y.to_numpy()).all(), (
    "Non-finite values were detected in the target variable."
)

print("\nModel input validation passed.")

print("Training features:", X.shape[1])
print("Records:", X.shape[0])


# ------------------------------------------------------------
# 8. Split training and test datasets
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE
)

print("\nTraining records:", f"{len(X_train):,}")
print("Testing records:", f"{len(X_test):,}")


# ------------------------------------------------------------
# 9. Train a shallow regression tree
# ------------------------------------------------------------

tree_model = DecisionTreeRegressor(
    max_depth=4,
    min_samples_leaf=2000,
    random_state=RANDOM_STATE
)

tree_model.fit(
    X_train,
    y_train
)

print("\nDecision Tree training completed.")


# ------------------------------------------------------------
# 10. Predict on the independent test subset
# ------------------------------------------------------------

predictions = tree_model.predict(
    X_test
)


# ------------------------------------------------------------
# 11. Evaluate model performance
# ------------------------------------------------------------

mae_log_duration = mean_absolute_error(
    y_test,
    predictions
)

r2_log_duration = r2_score(
    y_test,
    predictions
)

model_metrics = pd.DataFrame([
    {
        "MODEL": "Shallow Decision Tree",
        "SAMPLE_SIZE": sample_size,
        "TRAIN_RECORDS": len(X_train),
        "TEST_RECORDS": len(X_test),
        "MAX_DEPTH": 4,
        "MIN_SAMPLES_LEAF": 2000,
        "MAE_LOG_DURATION": mae_log_duration,
        "R2_LOG_DURATION": r2_log_duration
    }
])

export_table(
    model_metrics,
    "duration_tree_model_metrics.csv"
)


# ------------------------------------------------------------
# 12. Extract interpretable tree rules
# ------------------------------------------------------------

tree_rules = export_text(
    tree_model,
    feature_names=list(X.columns),
    decimals=3,
    max_depth=4
)

rules_path = (
    RULE_DIR
    / "duration_tree_rules.txt"
)

rules_path.write_text(
    tree_rules,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 13. Calculate feature importance
# ------------------------------------------------------------

feature_importance = pd.DataFrame({
    "FEATURE": X.columns,
    "IMPORTANCE": tree_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "IMPORTANCE",
    ascending=False
).reset_index(drop=True)

export_table(
    feature_importance,
    "duration_tree_feature_importance.csv"
)


# ------------------------------------------------------------
# 14. Display model evaluation
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("DECISION TREE MODEL EVALUATION")
print("=" * 65)

display(model_metrics)


# ------------------------------------------------------------
# 15. Display feature importance
# ------------------------------------------------------------

print("\nTop 15 Important Features:")
display(
    feature_importance.head(15)
)


# ------------------------------------------------------------
# 16. Display discovered tree conditions
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("DISCOVERED DECISION TREE CONDITIONS")
print("=" * 65)

print(tree_rules)


# ------------------------------------------------------------
# 17. Display output locations
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("OUTPUT FILES")
print("=" * 65)

print(
    "Model metrics:",
    TABLE_DIR / "duration_tree_model_metrics.csv"
)

print(
    "Feature importance:",
    TABLE_DIR / "duration_tree_feature_importance.csv"
)

print(
    "Tree rules:",
    rules_path
)

print("\nDecision Tree rule discovery completed.")
print("Original transaction records remain unchanged.")

In [ ]:
# ============================================================
# CELL 14 - GENERATE DURATION RULE REGISTER
# Purpose:
# Document candidate duration rules, evidence references,
# research limitations, and business implications.
# ============================================================

duration_candidates = [
    {
        "RULE_ID": "DR01",
        "RULE_FAMILY": "Routing Duration",
        "HYPOTHESIS": (
            "Different routing types have different "
            "payment duration distributions."
        ),
        "EVIDENCE": "duration_by_routing_type.csv",
        "BUSINESS_VALUE": "Routing performance benchmarking"
    },
    {
        "RULE_ID": "DR02",
        "RULE_FAMILY": "Routing Complexity",
        "HYPOTHESIS": (
            "Participant count is associated with "
            "payment duration."
        ),
        "EVIDENCE": "duration_by_routing_and_participants.csv",
        "BUSINESS_VALUE": "Identify complex slow-payment routes"
    },
    {
        "RULE_ID": "DR03",
        "RULE_FAMILY": "FX Duration",
        "HYPOTHESIS": (
            "FX conversion count remains associated with "
            "duration within comparable routing groups."
        ),
        "EVIDENCE": "fx_adjacent_duration_comparisons.csv",
        "BUSINESS_VALUE": "FX processing efficiency investigation"
    },
    {
        "RULE_ID": "DR04",
        "RULE_FAMILY": "Covered vs Indirect",
        "HYPOTHESIS": (
            "Covered and Indirect payments differ in "
            "duration after matching observed characteristics."
        ),
        "EVIDENCE": "indirect_covered_duration_matched_summary.csv",
        "BUSINESS_VALUE": "Covered routing performance investigation"
    },
    {
        "RULE_ID": "DR05",
        "RULE_FAMILY": "Path Observability",
        "HYPOTHESIS": (
            "Routing gaps remain associated with duration "
            "after controlling for routing complexity."
        ),
        "EVIDENCE": "gap_duration_matched_comparison.csv",
        "BUSINESS_VALUE": "Payment traceability monitoring"
    },
    {
        "RULE_ID": "DR06",
        "RULE_FAMILY": "Slow Payments",
        "HYPOTHESIS": (
            "Slow-payment frequency varies systematically "
            "across routing and participant groups."
        ),
        "EVIDENCE": "slow_payment_rates_by_route_participants.csv",
        "BUSINESS_VALUE": "Operational latency monitoring"
    },
    {
        "RULE_ID": "DR07",
        "RULE_FAMILY": "Direct Duration Regimes",
        "HYPOTHESIS": (
            "Direct payments contain distinct fast and slow "
            "duration behaviour not fully explained by FX."
        ),
        "EVIDENCE": "direct_duration_regime_summary.csv",
        "BUSINESS_VALUE": "Investigate slow Direct transactions"
    },
    {
        "RULE_ID": "DR08",
        "RULE_FAMILY": "Currency and Duration",
        "HYPOTHESIS": (
            "Currency is associated with duration differences "
            "within routing types."
        ),
        "EVIDENCE": "duration_by_route_currency.csv",
        "BUSINESS_VALUE": "Currency-specific performance monitoring"
    },
    {
        "RULE_ID": "DR09",
        "RULE_FAMILY": "Party Behaviour",
        "HYPOTHESIS": (
            "Party relationships are associated with "
            "different payment duration patterns."
        ),
        "EVIDENCE": "duration_by_route_party.csv",
        "BUSINESS_VALUE": "Payment segment performance analysis"
    },
    {
        "RULE_ID": "DR10",
        "RULE_FAMILY": "Temporal Behaviour",
        "HYPOTHESIS": (
            "Payment duration varies across UTC hours."
        ),
        "EVIDENCE": "duration_by_route_hour.csv",
        "BUSINESS_VALUE": "Temporal performance monitoring"
    }
]

duration_rule_register = pd.DataFrame(
    duration_candidates
)

duration_rule_register["STATUS"] = "NOT_VALIDATED"
duration_rule_register["EFFECT_SIZE"] = ""
duration_rule_register["SCOPE"] = ""
duration_rule_register["EXCEPTIONS"] = ""
duration_rule_register["CONFIDENCE"] = ""
duration_rule_register["FINAL_RULE"] = ""

register_path = (
    RULE_DIR / "payment_duration_candidate_rules.csv"
)

duration_rule_register.to_csv(
    register_path,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 65)
print("PAYMENT DURATION CANDIDATE RULE REGISTER")
print("=" * 65)

print("Candidate rules:", len(duration_rule_register))

display(duration_rule_register)

print("\nRule register generated successfully.")

In [ ]:
# ============================================================
# CELL 15 - FINAL DURATION ANALYSIS VALIDATION
# Purpose:
# Validate transaction coverage and core evidence tables.
# ============================================================

validation_checks = []


def add_validation(name, actual, expected):

    validation_checks.append({
        "CHECK": name,
        "ACTUAL": int(actual),
        "EXPECTED": int(expected),
        "PASSED": int(actual) == int(expected)
    })


# ------------------------------------------------------------
# 1. Validate full analytical dataset
# ------------------------------------------------------------

add_validation(
    "Compact analytical dataset",
    len(duration_df),
    EXPECTED_RECORDS
)

add_validation(
    "Global duration population",
    global_stats["TRANSACTIONS"],
    EXPECTED_RECORDS
)

# ------------------------------------------------------------
# 2. Validate group aggregations
# ------------------------------------------------------------

add_validation(
    "Duration by routing",
    route_duration["TRANSACTION_COUNT"].sum(),
    EXPECTED_RECORDS
)

add_validation(
    "Duration by routing and participants",
    participant_route_duration["TRANSACTION_COUNT"].sum(),
    EXPECTED_RECORDS
)

add_validation(
    "Duration by routing, participants and FX",
    fx_duration["TRANSACTION_COUNT"].sum(),
    EXPECTED_RECORDS
)

add_validation(
    "Duration by routing, participants, FX and gap",
    gap_duration["TRANSACTION_COUNT"].sum(),
    EXPECTED_RECORDS
)

add_validation(
    "Duration by currency and routing",
    currency_route_duration["TRANSACTION_COUNT"].sum(),
    EXPECTED_RECORDS
)

add_validation(
    "Duration by party and routing",
    party_route_duration["TRANSACTION_COUNT"].sum(),
    EXPECTED_RECORDS
)

add_validation(
    "Duration by UTC hour and routing",
    hour_duration["TRANSACTION_COUNT"].sum(),
    EXPECTED_RECORDS
)

# ------------------------------------------------------------
# 3. Validate known Direct population
# ------------------------------------------------------------

add_validation(
    "Direct payment records",
    len(direct_df),
    662_245
)

# ------------------------------------------------------------
# 4. Generate validation report
# ------------------------------------------------------------

validation_df = pd.DataFrame(validation_checks)

display(validation_df)

assert validation_df["PASSED"].all(), (
    "One or more duration validation checks failed."
)

export_table(
    validation_df,
    "duration_analysis_validation.csv"
)

# ------------------------------------------------------------
# 5. Display final summary
# ------------------------------------------------------------

print("=" * 65)
print("PAYMENT DURATION ANALYSIS VALIDATION")
print("=" * 65)

print("Transactions:", f"{EXPECTED_RECORDS:,}")
print("Duration candidates:", len(duration_rule_register))

print("Validation status: PASSED")
print("Source transactions were not modified.")

print("\nTables saved to:", TABLE_DIR)
print("Charts saved to:", CHART_DIR)
print("Rule register saved to:", register_path)

print("\nPayment duration investigation completed.")